## 카메라 연결

이제 학습한 AI를 웹캠에 연결한다.

05_camera_test.py

먼저 가장 간단하게 확인해보자

아래 코드 하나만 실행해봐.
이건 게임 기능을 전부 빼고 카메라 + AI 인식만 확인하는 코드야.

In [18]:
import cv2
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
import random


# ==================================================
# 1. CNN 모델 정의
# ==================================================

class CNN(nn.Module):

    def __init__(self, num_classes):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(64 * 16 * 16, 128),
            nn.ReLU(),

            nn.Linear(128, num_classes)
        )


    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x


# ==================================================
# 2. 학습한 모델 불러오기
# ==================================================

checkpoint = torch.load(
    "models/object_cnn.pth",
    map_location="cpu"
)

classes = checkpoint["classes"]

model = CNN(len(classes))

model.load_state_dict(
    checkpoint["model_state"]
)

model.eval()


# ==================================================
# 3. 이미지 전처리
# ==================================================

transform = transforms.Compose([

    transforms.Resize((128, 128)),

    transforms.ToTensor()
])


# ==================================================
# 4. 게임 설정
# ==================================================

# 정답으로 인정할 최소 확률
confidence_threshold = 0.95

# 몇 번 연속 맞히면 정답인지
required_count = 20

# 현재 정답 연속 횟수
correct_count = 0

# 지금까지 맞힌 물건
completed_objects = []


# ==================================================
# 5. 첫 번째 목표물 랜덤 선택
# ==================================================

remaining_objects = classes.copy()

target = random.choice(remaining_objects)

print()
print("================================")
print("       🔎 물건 찾기 게임")
print("================================")
print()

print("AI가 알고 있는 물건:")
print(classes)

print()

print("🎯 찾아야 하는 물건:")
print(f"👉 {target}")

print()


# ==================================================
# 6. 웹캠 실행
# ==================================================

camera = cv2.VideoCapture(0)

if not camera.isOpened():

    print("❌ 웹캠을 열 수 없습니다.")

    raise SystemExit

else:

    print("✅ 웹캠이 시작되었습니다.")


# ==================================================
# 7. 게임 진행
# ==================================================

while True:

    # --------------------------------------------------
    # 웹캠에서 프레임 가져오기
    # --------------------------------------------------

    ret, frame = camera.read()

    if not ret:

        print("❌ 웹캠 화면을 읽을 수 없습니다.")

        break


    # --------------------------------------------------
    # BGR → RGB
    # --------------------------------------------------

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )


    # --------------------------------------------------
    # PIL 이미지
    # --------------------------------------------------

    image = Image.fromarray(rgb)


    # --------------------------------------------------
    # 이미지 크기 변경
    # --------------------------------------------------

    image = transform(image)


    # --------------------------------------------------
    # 배치 차원 추가
    # --------------------------------------------------

    image = image.unsqueeze(0)


    # --------------------------------------------------
    # AI 예측
    # --------------------------------------------------

    with torch.no_grad():

        output = model(image)

        probability = torch.softmax(
            output,
            dim=1
        )

        confidence, predicted = torch.max(
            probability,
            dim=1
        )


    # --------------------------------------------------
    # 예측 결과
    # --------------------------------------------------

    label = classes[predicted.item()]

    confidence = confidence.item()


    # --------------------------------------------------
    # 정답 판정
    # --------------------------------------------------

    if label == target and confidence >= confidence_threshold:

        correct_count += 1

    else:

        correct_count = 0


    # ==================================================
    # 화면 표시
    # ==================================================

    cv2.putText(

        frame,

        f"Target: {target}",

        (30, 40),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.8,

        (0, 255, 255),

        2
    )


    cv2.putText(

        frame,

        f"AI: {label}",

        (30, 80),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.8,

        (0, 255, 0),

        2
    )


    cv2.putText(

        frame,

        f"Confidence: {confidence * 100:.1f}%",

        (30, 120),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.8,

        (0, 255, 0),

        2
    )


    cv2.putText(

        frame,

        f"Correct: {correct_count}/{required_count}",

        (30, 160),

        cv2.FONT_HERSHEY_SIMPLEX,

        0.8,

        (255, 255, 255),

        2
    )


    # ==================================================
    # 정답!
    # ==================================================

    if correct_count >= required_count:

        # 현재 물건을 완료 목록에 추가
        completed_objects.append(target)

        # 완료된 물건을 남은 목록에서 제거
        remaining_objects.remove(target)


        # --------------------------------------------------
        # 성공 메시지
        # --------------------------------------------------

        cv2.putText(

            frame,

            "CORRECT!",

            (100, 230),

            cv2.FONT_HERSHEY_SIMPLEX,

            1.2,

            (0, 255, 0),

            3
        )


        cv2.imshow(
            "Object Game",
            frame
        )

        cv2.waitKey(1500)


        print()
        print("🎉 정답!")
        print(f"찾은 물건: {target}")
        print()


        # --------------------------------------------------
        # 모든 물건을 찾았는지 확인
        # --------------------------------------------------

        if len(remaining_objects) == 0:

            print("================================")
            print("       🎉 게임 클리어! 🎉")
            print("================================")
            print()

            print("찾은 물건:")
            print(completed_objects)

            break


        # --------------------------------------------------
        # 다음 물건 선택
        # --------------------------------------------------

        target = random.choice(remaining_objects)

        # 정답 횟수 초기화
        correct_count = 0


        print("================================")
        print("🎯 다음 물건:")
        print(f"👉 {target}")
        print("================================")


        continue


    # ==================================================
    # 웹캠 화면 출력
    # ==================================================

    cv2.imshow(
        "Object Game",
        frame
    )


    # ==================================================
    # Q 종료
    # ==================================================

    if cv2.waitKey(1) & 0xFF == ord("q"):

        print()
        print("게임을 종료했습니다.")

        break


# ==================================================
# 8. 웹캠 종료
# ==================================================

camera.release()

cv2.destroyAllWindows()

print()
print("웹캠 종료")


       🔎 물건 찾기 게임

AI가 알고 있는 물건:
['nipper', 'pen', 'wire stripper']

🎯 찾아야 하는 물건:
👉 pen

✅ 웹캠이 시작되었습니다.

🎉 정답!
찾은 물건: pen

🎯 다음 물건:
👉 nipper

🎉 정답!
찾은 물건: nipper

🎯 다음 물건:
👉 wire stripper

🎉 정답!
찾은 물건: wire stripper

       🎉 게임 클리어! 🎉

찾은 물건:
['pen', 'nipper', 'wire stripper']

웹캠 종료
